# FARUMASI Pharmacist Assistant: full training and evaluation pipeline

Runtime → Change runtime type → **T4 GPU**. Run the cells in order (about 60–90 minutes in total).

Stages: data download → preparation → intent classifier (baseline + DistilBERT) → retrieval (BM25, MiniLM zero-shot, fine-tuned MiniLM, hybrid) → answer generation (extractive, Flan-T5 zero-shot, Flan-T5 + LoRA) → safety check → summary tables → (optional) deploy to Hugging Face Spaces.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
REPO_URL = "https://github.com/<your-github-username>/farumasi-pharmacist-qa.git"  # <- edit
!git clone -q $REPO_URL repo
%cd repo
!pip install -q -r requirements.txt

## 1. Data

In [ ]:
!python -m pharmqa.data.download
!python -m pharmqa.data.prepare
!python -m pharmqa.data.intent_data

## 2. Intent / scope classifier
Baseline TF-IDF + logistic regression (4 configs), then DistilBERT trained on the original data and on the data augmented with real patient questions.

In [ ]:
!python -m pharmqa.intent.baseline
!python -m pharmqa.intent.transformer --train_file train --name distilbert_original
!python -m pharmqa.intent.transformer --train_file train_consumer --name distilbert

## 3. Retrieval

In [ ]:
!python -m pharmqa.retrieval.bm25
!python -m pharmqa.retrieval.dense --mode all --epochs 2 --batch 64 --lr 2e-5

Rwanda FDA experiment (before): same retriever trained **without** the Rwanda leaflet questions. Compare the `RwandaFDA-PIL` rows in `results/retrieval_minilm_finetuned*.json`. Saved separately; the deployed retriever is untouched.

In [ ]:
!python -m pharmqa.retrieval.dense --mode finetune --epochs 2 --batch 64 --lr 2e-5 --exclude_source RwandaFDA-PIL

## 4. Answer generation (Flan-T5-base + LoRA)

In [ ]:
!python -m pharmqa.generation.train_lora --epochs 3 --lr 3e-4 --rank 16 --batch 16

In [ ]:
for system in ["extractive", "zeroshot", "lora"]:
    for context in ["gold", "retrieved"]:
        !python -m pharmqa.generation.evaluate --system {system} --context {context} --n 400

Rwanda FDA experiment (before) for the reader: LoRA trained without Rwanda leaflet questions, evaluated on the same test questions.

In [ ]:
!python -m pharmqa.generation.train_lora --epochs 3 --lr 3e-4 --rank 16 --batch 16 --exclude_source RwandaFDA-PIL --out artifacts/generator_flant5_lora_no_rwandafda
!python -m pharmqa.generation.evaluate --system lora --context gold --n 400 --adapter artifacts/generator_flant5_lora_no_rwandafda --tag _no_rwandafda

## 5. Safety check and summary tables

In [ ]:
!python -m pharmqa.safety_eval
!python -m pharmqa.report_tables

## 6. Save results and models
Downloads a zip with `results/` (commit these to GitHub) and `artifacts/` (trained models).

In [ ]:
!zip -qr results.zip results data/processed/dataset_stats.json data/processed/intent_stats.json
!zip -qr artifacts.zip artifacts -x 'artifacts/*/checkpoints/*'
from google.colab import files
files.download("results.zip")
files.download("artifacts.zip")

## 7. (Optional) Deploy to Hugging Face Spaces
Create a token with *write* access at https://huggingface.co/settings/tokens.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()
SPACE = "<your-hf-username>/farumasi-pharmacist-qa"  # <- edit
!python scripts/deploy_space.py --space $SPACE